# CV ↔ Job Posting Skill-Gap Analysis — NER pipeline (pretrained DistilBERT)

This notebook rebuilds the skill-gap model around an explicit pipeline instead of a single
multi-label classifier, and fine-tunes a **pretrained DistilBERT** for the skill-extraction step
(swapping out the earlier from-scratch BiLSTM / Transformer taggers).

```
Job postings ──▶ Extract skills ──▶ Role database
                                          │
CV ───────────▶ Extract skills           │
        │                                │
        └───────────────▶ Compare ◀──────┘
                             │
                        Missing skills
                             │
                        Recommendations
```

**Pipeline stages implemented below:**
1. Load job postings and turn their existing skill annotations into token-level BIO tags (distant supervision).
2. Fine-tune `distilbert-base-uncased` (via Hugging Face `transformers`, `AutoModelForTokenClassification`) as
   a token classifier that extracts skill spans, typed as TECH / SOFT / TOOL / DOMAIN / CERT.
3. Run the model over job postings and aggregate the results per job title → **role database**.
4. Run the *same* model over a candidate's CV → the CV's skill set.
5. Compare CV skills vs. the target role's required skills → missing skills → recommendations.

> **Network note:** fine-tuning `distilbert-base-uncased` requires downloading its pretrained weights
> and tokenizer from `huggingface.co` the first time. This sandboxed environment does **not** have
> network access to `huggingface.co`, so the download cell below will fail here — that's expected and
> is not a bug in the code. Run this notebook somewhere with internet access (your own machine, Colab,
> Kaggle, a training server) and it will download the ~268MB model automatically on first run, or
> pre-download it once with:
> ```bash
> huggingface-cli download distilbert-base-uncased
> ```
> A GPU is strongly recommended — fine-tuning DistilBERT over ~7k postings on CPU will be slow
> (expect tens of minutes per epoch on CPU vs. under a minute per epoch on a modern GPU).


In [2]:
import pandas as pd
import numpy as np
import re
import json
from collections import Counter

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForTokenClassification

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

torch.manual_seed(42)
np.random.seed(42)


Device: cuda


## Stage 1 — Load job postings

Each posting has skills split into five categories (`skills_technical`, `skills_soft`, `skills_tool`,
`skills_domain`, `skills_certification`). We keep the category split — it becomes the entity *type*
in the NER tag set.

In [ ]:
df = pd.read_csv('../data/processed/data_tech_only.csv')
df = df[df["skill_count"] > 0].dropna(subset=["cleaned_text"]).reset_index(drop=True)
print(df.shape)
df.head()


(6850, 11)


,row_index,title,cleaned_text,skills_technical,skills_soft,skills_tool,skills_domain,skills_certification,all_skills,skill_count,cleaned_title
0,0,10 + Blockchain Nodes / Masternodes to set up,blockchain nodes masternodes set requirements ...,blockchain; Kyber Network; Nebulas; SecretNetw...,NaN,NaN,crypto,NaN,blockchain; crypto; Kyber Network; Nebulas; Se...,16,blockchain nodes masternodes set
1,1,10 .NET Developers (Middle and Senior level),net developers middle senior level greetings n...,.NET; microservices; multi-cloud; SaaS,NaN,NaN,NaN,NaN,.NET; microservices; multi-cloud; SaaS,4,net developers middle senior level
2,2,"10X Engineer (co-founder, #4 employee, USD 11-...",x engineer co founder employee usd k equity pr...,live video chat; co-browsing,NaN,NaN,NaN,NaN,live video chat; co-browsing,2,x engineer co founder employee usd k equity
3,5,1806/01 PHP Developer,php developer requirements : experience php ye...,PHP; Laravel; JavaScript; Vue.js; MySQL; serve...,NaN,Git,NaN,NaN,PHP; Laravel; JavaScript; Vue.js; MySQL; Git; ...,7,php developer
4,9,1812/35 Middle DataBase developer,middle database developer job responsibilities...,SQL; database development; stored procedures; ...,NaN,SQL Profiler,NaN,NaN,SQL; database development; stored procedures; ...,7,middle database developer


## Stage 2 — Distant-supervision BIO tagging

There's no span-level annotation in the source data, only a list of skill phrases per posting. We
turn that into word-level BIO tags: split each posting into words, then for every known skill phrase
find it as a word subsequence and tag it `B-<TYPE>` / `I-<TYPE>` (longest phrases matched first so a
multi-word skill isn't partially swallowed by a shorter one).

Tag set: `O` plus `B-`/`I-` for `TECH`, `SOFT`, `TOOL`, `DOMAIN`, `CERT` (11 tags total). This word-level
tagging is done with a simple regex tokenizer — DistilBERT's own (subword) tokenizer is applied later,
with the word-level tags realigned to its subword tokens.

In [5]:
CATEGORY_COLS = {
    "TECH": "skills_technical",
    "SOFT": "skills_soft",
    "TOOL": "skills_tool",
    "DOMAIN": "skills_domain",
    "CERT": "skills_certification",
}

def parse_skills(cell):
    if pd.isna(cell):
        return []
    return [s.strip() for s in str(cell).split(";") if s.strip()]

for col in CATEGORY_COLS.values():
    df[col + "_list"] = df[col].apply(parse_skills)

def tokenize(text):
    # Word-level tokenizer used only to build ground-truth BIO tags (not fed to the model).
    return re.findall(r"[a-z0-9+#.]+", str(text).lower())

TAG_LIST = ["O"]
for cat in CATEGORY_COLS:
    TAG_LIST += [f"B-{cat}", f"I-{cat}"]
tag_to_idx = {t: i for i, t in enumerate(TAG_LIST)}
id2label = {i: t for i, t in enumerate(TAG_LIST)}
label2id = {t: i for i, t in enumerate(TAG_LIST)}
O_IDX = tag_to_idx["O"]
NUM_TAGS = len(TAG_LIST)
print("Tags:", TAG_LIST)

def find_span(tokens, phrase_tokens, taken):
    n, m = len(tokens), len(phrase_tokens)
    if m == 0:
        return None
    for i in range(n - m + 1):
        if any(taken[i:i + m]):
            continue
        if tokens[i:i + m] == phrase_tokens:
            return i
    return None

def build_bio_tags(text, skills_by_cat):
    # Distant-supervision BIO tagging: match each known skill phrase to a word span.
    tokens = tokenize(text)
    tags = [O_IDX] * len(tokens)
    taken = [False] * len(tokens)

    ordered = [(cat, s) for cat, skills in skills_by_cat.items() for s in skills]
    ordered.sort(key=lambda x: -len(tokenize(x[1])))  # longest phrase first

    for cat, skill in ordered:
        phrase_tokens = tokenize(skill)
        start = find_span(tokens, phrase_tokens, taken)
        if start is None:
            continue
        end = start + len(phrase_tokens)
        tags[start] = tag_to_idx[f"B-{cat}"]
        for i in range(start + 1, end):
            tags[i] = tag_to_idx[f"I-{cat}"]
        for i in range(start, end):
            taken[i] = True
    return tokens, tags

all_tokens, all_tags = [], []
for _, row in df.iterrows():
    skills_by_cat = {cat: row[col + "_list"] for cat, col in CATEGORY_COLS.items()}
    toks, tags = build_bio_tags(row["cleaned_text"], skills_by_cat)
    all_tokens.append(toks)
    all_tags.append(tags)

df["tokens"] = all_tokens
df["bio_tags"] = all_tags

covered = sum(any(t != O_IDX for t in tags) for tags in all_tags)
print(f"Postings with >=1 tagged skill span: {covered}/{len(df)}")


Tags: ['O', 'B-TECH', 'I-TECH', 'B-SOFT', 'I-SOFT', 'B-TOOL', 'I-TOOL', 'B-DOMAIN', 'I-DOMAIN', 'B-CERT', 'I-CERT']
Postings with >=1 tagged skill span: 6205/6850


## Train / val / test split

In [6]:
idx_train, idx_temp = train_test_split(df.index, test_size=0.3, random_state=42)
idx_val, idx_test = train_test_split(idx_temp, test_size=0.5, random_state=42)
print(f"train: {len(idx_train)}  val: {len(idx_val)}  test: {len(idx_test)}")


train: 4795  val: 1027  test: 1028


## DistilBERT tokenizer + subword label alignment

DistilBERT splits words into subword pieces (e.g. `"kubernetes"` → `["ku", "##bernetes"]`), so our
word-level BIO tags need to be re-mapped onto its subword tokens. Standard approach: feed the
pre-split word list in with `is_split_into_words=True`, then use `word_ids()` to find which original
word each subword token came from. Only the **first** subword of each word keeps the real tag; the
rest (and special/padding tokens) get `-100`, which the loss ignores.

In [7]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 100
PAD_TAG = -100  # ignored by the loss

try:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
except Exception as e:
    print(
        "Could not download the pretrained tokenizer/model from huggingface.co.\n"
        "This environment has no network access to huggingface.co, so this is expected here.\n"
        "Run this notebook somewhere with internet access (local machine, Colab, Kaggle, a "
        "training server) -- it will download distilbert-base-uncased automatically, or "
        "pre-download it once with:  huggingface-cli download distilbert-base-uncased\n"
    )
    raise

def align_labels_with_tokens(word_ids, word_tags):
    labels = []
    prev_word_id = None
    for wid in word_ids:
        if wid is None:
            labels.append(PAD_TAG)
        elif wid != prev_word_id:
            labels.append(word_tags[wid])   # first subword of this word gets the real tag
        else:
            labels.append(PAD_TAG)          # continuation subwords are ignored by the loss
        prev_word_id = wid
    return labels


## Dataset & DataLoaders

In [8]:
class SkillNERDataset(Dataset):
    def __init__(self, indices):
        self.indices = list(indices)

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, i):
        row = df.loc[self.indices[i]]
        enc = tokenizer(
            row["tokens"], is_split_into_words=True, truncation=True,
            max_length=MAX_LEN, padding="max_length", return_tensors="pt",
        )
        word_ids = enc.word_ids(batch_index=0)
        labels = align_labels_with_tokens(word_ids, row["bio_tags"])
        return {
            "input_ids": enc["input_ids"][0],
            "attention_mask": enc["attention_mask"][0],
            "labels": torch.tensor(labels, dtype=torch.long),
        }

BATCH_SIZE = 16
train_loader = DataLoader(SkillNERDataset(idx_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(SkillNERDataset(idx_val), batch_size=BATCH_SIZE)
test_loader  = DataLoader(SkillNERDataset(idx_test), batch_size=BATCH_SIZE)


## Model — pretrained DistilBERT token classifier

`AutoModelForTokenClassification` loads `distilbert-base-uncased`'s pretrained encoder and attaches a
fresh linear head with `num_labels=11` (our BIO tag set) on top. Fine-tuning updates the whole
network (encoder + head), so the model adapts its pretrained language understanding to this task.

In [9]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_TAGS, id2label=id2label, label2id=label2id,
).to(device)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForTokenClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Trainable params: 66,371,339


## Entity-level evaluation

We score whole skill **spans** (boundaries + type must match exactly), not individual subword tags.
Since only the first subword of each word carries a real label (the rest are `-100`), gathering the
non-`-100` positions in order reconstructs the original word-level tag sequence.

In [10]:
def tags_to_spans(tags):
    spans = []
    start, cat = None, None
    for i, t in enumerate(list(tags) + [O_IDX]):
        label = TAG_LIST[t] if t != PAD_TAG else "O"
        if label.startswith("B-"):
            if start is not None:
                spans.append((start, i, cat))
            start, cat = i, label[2:]
        elif label.startswith("I-") and cat == label[2:]:
            continue
        else:
            if start is not None:
                spans.append((start, i, cat))
            start, cat = None, None
    return spans

def entity_prf(all_pred_tags, all_true_tags):
    tp = fp = fn = 0
    for pred, true in zip(all_pred_tags, all_true_tags):
        pred_spans = set(tags_to_spans(pred))
        true_spans = set(tags_to_spans(true))
        tp += len(pred_spans & true_spans)
        fp += len(pred_spans - true_spans)
        fn += len(true_spans - pred_spans)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


## Fine-tuning loop

Same early-stopping-on-validation-F1 pattern as before, using a small learning rate (`2e-5`) typical
for fine-tuning pretrained transformers, and the HF model's built-in loss (computed when `labels` is
passed) which already applies `ignore_index=-100`.

In [11]:
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, n_examples = 0.0, 0
    pred_seqs, true_seqs = [], []

    with torch.set_grad_enabled(train):
        for batch in loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * input_ids.size(0)
            n_examples += input_ids.size(0)

            preds = outputs.logits.argmax(-1)
            for p_row, y_row in zip(preds.cpu().tolist(), labels.cpu().tolist()):
                idxs = [j for j, t in enumerate(y_row) if t != PAD_TAG]
                pred_seqs.append([p_row[j] for j in idxs])
                true_seqs.append([y_row[j] for j in idxs])

    precision, recall, f1 = entity_prf(pred_seqs, true_seqs)
    return total_loss / max(n_examples, 1), precision, recall, f1


EPOCHS = 4
PATIENCE = 2
best_val_f1 = -1
patience_counter = 0

for epoch in range(1, EPOCHS + 1):
    train_loss, *_ = run_epoch(train_loader, train=True)
    val_loss, val_p, val_r, val_f1 = run_epoch(val_loader, train=False)

    print(f"Epoch {epoch:02d} | train_loss {train_loss:.4f} | "
          f"val_loss {val_loss:.4f} val_precision {val_p:.3f} val_recall {val_r:.3f} val_f1 {val_f1:.3f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        model.save_pretrained("skill_ner_distilbert_best")
        tokenizer.save_pretrained("skill_ner_distilbert_best")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print("Early stopping — validation F1 stopped improving.")
            break


Epoch 01 | train_loss 0.3825 | val_loss 0.2681 val_precision 0.685 val_recall 0.469 val_f1 0.557


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 02 | train_loss 0.2247 | val_loss 0.2243 val_precision 0.660 val_recall 0.582 val_f1 0.618


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 03 | train_loss 0.1799 | val_loss 0.2144 val_precision 0.679 val_recall 0.612 val_f1 0.643


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch 04 | train_loss 0.1485 | val_loss 0.2181 val_precision 0.727 val_recall 0.582 val_f1 0.646


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [12]:
model = AutoModelForTokenClassification.from_pretrained("skill_ner_distilbert_best").to(device)
test_loss, test_p, test_r, test_f1 = run_epoch(test_loader, train=False)
print(f"TEST — precision {test_p:.3f} | recall {test_r:.3f} | f1 {test_f1:.3f}")


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

TEST — precision 0.724 | recall 0.583 | f1 0.646


## `extract_skills()` — the reusable "Extract skills" step

Used for **both** arms of the pipeline: extracting skills from job postings (to build the role
database) and extracting skills from a CV. Predictions are made on DistilBERT's subword tokens, then
mapped back to whole words via `word_ids()` before decoding into skill spans.

In [13]:
def extract_skills(text):
    words = tokenize(text)
    if not words:
        return []

    enc = tokenizer(
        words, is_split_into_words=True, truncation=True,
        max_length=MAX_LEN, padding="max_length", return_tensors="pt",
    )
    word_ids = enc.word_ids(batch_index=0)

    model.eval()
    with torch.no_grad():
        logits = model(
            input_ids=enc["input_ids"].to(device),
            attention_mask=enc["attention_mask"].to(device),
        ).logits
    preds = logits.argmax(-1)[0].cpu().tolist()

    word_tags = [O_IDX] * len(words)
    seen = set()
    for pos, wid in enumerate(word_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        word_tags[wid] = preds[pos]

    spans = tags_to_spans(word_tags)
    return [{"skill": " ".join(words[s:e]), "type": cat} for s, e, cat in spans]


extract_skills("experienced python developer with docker, kubernetes and aws")


[{'skill': 'python', 'type': 'TECH'},
 {'skill': 'docker', 'type': 'TOOL'},
 {'skill': 'kubernetes', 'type': 'TECH'},
 {'skill': 'aws', 'type': 'TECH'}]

## Stage 3 — Job postings → Extract skills → Role database

Run `extract_skills` over every posting and aggregate the results by job title. For each role, the
database stores the most frequently required skills — this is the reference each CV gets compared
against.

In [14]:
def build_role_database(dataframe, title_col="cleaned_title", top_k=15, min_postings=1):
    role_db = {}
    for role, group in dataframe.groupby(title_col):
        if len(group) < min_postings:
            continue
        counter = Counter()
        for text in group["cleaned_text"]:
            for item in extract_skills(text):
                counter[(item["skill"], item["type"])] += 1
        top = counter.most_common(top_k)
        role_db[role] = [{"skill": s, "type": t, "freq": f} for (s, t), f in top]
    return role_db


# Build the role database from postings the model was trained/validated on
role_db = build_role_database(df.loc[idx_train.union(idx_val)])
print(f"Roles in database: {len(role_db)}")
sample_role = max(role_db, key=lambda r: sum(s['freq'] for s in role_db[r]))
print(f"\nExample — '{sample_role}':")
for s in role_db[sample_role]:
    print(f"  {s['skill']:<25} ({s['type']})  seen in {s['freq']} posting(s)")


Roles in database: 2545

Example — 'android developer':
  android                   (TECH)  seen in 318 posting(s)
  kotlin                    (TECH)  seen in 117 posting(s)
  java                      (TECH)  seen in 104 posting(s)
  android sdk               (TECH)  seen in 47 posting(s)
  oop                       (TECH)  seen in 18 posting(s)
  git                       (TOOL)  seen in 17 posting(s)
  rxjava                    (TECH)  seen in 14 posting(s)
  dagger                    (TECH)  seen in 14 posting(s)
  retrofit                  (TECH)  seen in 13 posting(s)
  fintech                   (DOMAIN)  seen in 12 posting(s)
  computer science          (DOMAIN)  seen in 9 posting(s)
  coroutines                (TECH)  seen in 8 posting(s)
  android jetpack           (TECH)  seen in 7 posting(s)
  design patterns           (TECH)  seen in 7 posting(s)
  android studio            (TOOL)  seen in 7 posting(s)


## Stage 4 — CV → Extract skills

In [15]:
cv_text = """
Experienced backend developer with 4 years building REST APIs in Python and Node.js.
Strong SQL and PostgreSQL skills, some exposure to Docker and Git-based workflows.
"""

cv_skills = extract_skills(cv_text)
cv_skills


[{'skill': 'rest apis', 'type': 'TECH'},
 {'skill': 'python', 'type': 'TECH'},
 {'skill': 'sql', 'type': 'TECH'},
 {'skill': 'postgresql', 'type': 'TECH'},
 {'skill': 'docker', 'type': 'TOOL'},
 {'skill': 'git', 'type': 'TOOL'}]

## Stage 5 — Compare → Missing skills → Recommendations

In [16]:
def compare_to_role(cv_text, role_name, role_db, top_n=10):
    cv_skill_set = {item["skill"] for item in extract_skills(cv_text)}
    required = [s["skill"] for s in role_db.get(role_name, [])[:top_n]]

    have = [s for s in required if s in cv_skill_set]
    missing = [s for s in required if s not in cv_skill_set]
    match_score = round(100 * len(have) / len(required)) if required else 0

    return {"role": role_name, "have": have, "missing": missing, "match_score": match_score}


def recommend(gap_result):
    if not gap_result["missing"]:
        return [f"Great fit — you already cover the top skills for '{gap_result['role']}'."]
    return [f"Consider learning or highlighting: {s}" for s in gap_result["missing"]]


gap = compare_to_role(cv_text, sample_role, role_db)
print(f"Target role: {gap['role']}")
print(f"Match score: {gap['match_score']}%")
print(f"Have:    {gap['have']}")
print(f"Missing: {gap['missing']}")
print()
for line in recommend(gap):
    print("-", line)


Target role: android developer
Match score: 10%
Have:    ['git']
Missing: ['android', 'kotlin', 'java', 'android sdk', 'oop', 'rxjava', 'dagger', 'retrofit', 'fintech']

- Consider learning or highlighting: android
- Consider learning or highlighting: kotlin
- Consider learning or highlighting: java
- Consider learning or highlighting: android sdk
- Consider learning or highlighting: oop
- Consider learning or highlighting: rxjava
- Consider learning or highlighting: dagger
- Consider learning or highlighting: retrofit
- Consider learning or highlighting: fintech


## Save artifacts

Fine-tuned model + tokenizer, tag list, and the role database — everything needed to run the
pipeline (extract → compare → recommend) again without retraining.

In [17]:
import os
os.makedirs("../Model", exist_ok=True)

model.save_pretrained("../Model/skill_ner_distilbert")
tokenizer.save_pretrained("../Model/skill_ner_distilbert")

with open("../Model/tags.json", "w") as f:
    json.dump(TAG_LIST, f)

with open("../Model/role_database.json", "w") as f:
    json.dump(role_db, f, indent=2)

print("Saved: ../Model/skill_ner_distilbert/ (model + tokenizer), tags.json, role_database.json")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved: ../Model/skill_ner_distilbert/ (model + tokenizer), tags.json, role_database.json
